In [2]:
import pandas as pd
from rouge_score import rouge_scorer
from IPython.display import display, HTML

# =======================================================================
# CONFIGURAÇÕES
# =======================================================================

CAMINHO_INDICES = "./data/qmsum/indices_baseline.csv"
CAMINHO_QMSUM_BASELINE = "./hierarq_data/qmsum/resultados_baseline_templateB.csv"
CAMINHO_QMSUM_FT = "./hierarq_data/qmsum/resultados_finetuning.csv"

CAMINHOS_HIERARQUICO = {
    "Hier. Base": "./hierarq_data/hierarquico/resultados_base.csv",
    "Hier. SAMSum FT": "./hierarq_data/hierarquico/resultados_samsum.csv",
    "Hier. MediaSum FT": "./hierarq_data/hierarquico/resultados_mediasum.csv",
    "Hier. SAMSum+QMSum FT": "./hierarq_data/hierarquico/resultados_samsum_qmsumft.csv",
}

# =======================================================================
# CARREGAMENTO
# =======================================================================

def carregar_csv(caminho, sep=","):
    return pd.read_csv(caminho, sep=sep, engine="python",
                       quotechar='"', on_bad_lines="skip")

indices = carregar_csv(CAMINHO_INDICES)["indices"].tolist()
print(f"Total de índices selecionados: {len(indices)}")

df_qmsum_baseline = carregar_csv(CAMINHO_QMSUM_BASELINE)
df_qmsum_ft = carregar_csv(CAMINHO_QMSUM_FT)

dfs_hierarquico = {}
for nome, caminho in CAMINHOS_HIERARQUICO.items():
    dfs_hierarquico[nome] = carregar_csv(caminho)

print(f"\nLinhas QMSum Baseline (completo): {len(df_qmsum_baseline)}")
print(f"Linhas QMSum FT (completo): {len(df_qmsum_ft)}")

# =======================================================================
# ROUGE-L INDIVIDUAL
# =======================================================================

scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def calcular_rouge_l(referencia, gerado):
    if pd.isna(gerado) or not str(gerado).strip():
        return 0.0
    return scorer.score(str(referencia), str(gerado))["rougeL"].fmeasure

# =======================================================================
# MONTAGEM DA TABELA COMPARATIVA
# =======================================================================

# Pega as 40 amostras do baseline pelos índices originais
df_base_40 = df_qmsum_baseline.iloc[indices].reset_index(drop=True)

linhas_finais = []

for pos in range(len(df_base_40)):
    input_texto = df_base_40.iloc[pos]["input_texto"]
    referencia = df_base_40.iloc[pos]["resumo_referencia"]

    linha = {
        "input_texto": input_texto,
        "resumo_referencia": referencia,
    }

    # QMSum Baseline
    gerado = df_base_40.iloc[pos]["resumo_gerado"]
    linha["QMSum Baseline"] = gerado
    linha["QMSum Baseline_rouge"] = calcular_rouge_l(referencia, gerado)

    # QMSum FT — busca pelo resumo_referencia no CSV completo
    match = df_qmsum_ft[df_qmsum_ft["resumo_referencia"] == referencia]
    if len(match) > 0:
        gerado = match.iloc[0]["resumo_gerado"]
        linha["QMSum FT"] = gerado
        linha["QMSum FT_rouge"] = calcular_rouge_l(referencia, gerado)
    else:
        linha["QMSum FT"] = "N/A — não encontrado"
        linha["QMSum FT_rouge"] = None

    # Variantes hierárquicas — busca pelo resumo_referencia
    for nome, df in dfs_hierarquico.items():
        match = df[df["resumo_referencia"] == referencia]
        if len(match) > 0:
            gerado = match.iloc[0]["resumo_final"]
            linha[nome] = gerado
            linha[f"{nome}_rouge"] = calcular_rouge_l(referencia, gerado)
        else:
            linha[nome] = "N/A — não processado"
            linha[f"{nome}_rouge"] = None

    linhas_finais.append(linha)

df_comparativo = pd.DataFrame(linhas_finais)
print(f"\nTabela comparativa montada: {len(df_comparativo)} instâncias")

# Verifica quantas variantes hierárquicas tiveram correspondência
for nome in CAMINHOS_HIERARQUICO.keys():
    encontrados = (df_comparativo[f"{nome}_rouge"].notna()).sum()
    print(f"  {nome}: {encontrados}/{len(df_comparativo)} encontrados")

Total de índices selecionados: 40

Linhas QMSum Baseline (completo): 281
Linhas QMSum FT (completo): 281

Tabela comparativa montada: 40 instâncias
  Hier. Base: 40/40 encontrados
  Hier. SAMSum FT: 40/40 encontrados
  Hier. MediaSum FT: 40/40 encontrados
  Hier. SAMSum+QMSum FT: 40/40 encontrados


In [3]:
CORES_VARIANTES = {
    "QMSum Baseline": ("#bbdefb", "#0d47a1"),
    "QMSum FT": ("#ede7f6", "#4527a0"),
    "Hier. Base": ("#f0f0f0", "#444444"),
    "Hier. SAMSum FT": ("#e1f5ee", "#085041"),
    "Hier. MediaSum FT": ("#faece7", "#712b13"),
    "Hier. SAMSum+QMSum FT": ("#fbeaf0", "#72243e"),
}

def faixa(rouge, df, col):
    valores = df[col].dropna()
    if rouge < valores.quantile(0.33):
        return "BAIXO", "#ffcdd2", "#b71c1c"
    elif rouge < valores.quantile(0.66):
        return "MÉDIO", "#fff9c4", "#e65100"
    return "ALTO", "#c8e6c9", "#1b5e20"

def visualizar_comparativo_hierarquico(df):
    for i, row in df.iterrows():

        html = f"""
        <div style="
            border: 1px solid #ccc;
            border-radius: 8px;
            padding: 16px;
            margin-bottom: 32px;
            font-family: sans-serif;
            background: #ffffff;
        ">
            <div style="
                font-size: 16px; font-weight: 600; color: #111111;
                margin-bottom: 16px;
            ">
                Instância {i+1} de {len(df)}
            </div>

            <div style="margin-bottom: 12px;">
                <div style="
                    font-size: 11px; font-weight: 700;
                    text-transform: uppercase; color: #444444;
                    margin-bottom: 4px;
                ">Texto base (transcrição + query)</div>
                <div style="
                    background: #f0f0f0; border-radius: 4px;
                    padding: 10px; font-size: 12px;
                    line-height: 1.6; color: #111111;
                    max-height: 200px; overflow-y: auto;
                    white-space: pre-wrap;
                ">{row['input_texto']}</div>
            </div>

            <div style="margin-bottom: 12px;">
                <div style="
                    font-size: 11px; font-weight: 700;
                    text-transform: uppercase; color: #444444;
                    margin-bottom: 4px;
                ">Resumo de referência (gabarito)</div>
                <div style="
                    background: #c8e6c9; border-radius: 4px;
                    padding: 10px; font-size: 13px;
                    line-height: 1.6; color: #1b5e20;
                    white-space: pre-wrap;
                ">{row['resumo_referencia']}</div>
            </div>
        """

        for nome, (bg, fg) in CORES_VARIANTES.items():
            rouge_val = row[f"{nome}_rouge"]

            if rouge_val is not None:
                f, badge_bg, badge_fg = faixa(rouge_val, df, f"{nome}_rouge")
                badge = (
                    f'<span style="background:{badge_bg};color:{badge_fg};'
                    f'padding:3px 8px;border-radius:10px;font-size:11px;'
                    f'font-weight:600;">ROUGE-L: {rouge_val:.4f} — {f}</span>'
                )
            else:
                badge = ""

            html += f"""
            <div style="margin-bottom: 12px;">
                <div style="
                    display: flex; justify-content: space-between;
                    align-items: center; margin-bottom: 4px;
                ">
                    <div style="
                        font-size: 11px; font-weight: 700;
                        text-transform: uppercase; color: #444444;
                    ">{nome}</div>
                    {badge}
                </div>
                <div style="
                    background: {bg}; border-radius: 4px;
                    padding: 10px; font-size: 13px;
                    line-height: 1.6; color: {fg};
                    white-space: pre-wrap;
                ">{row[nome]}</div>
            </div>
            """

        html += "</div>"
        display(HTML(html))

# =======================================================================
# EXECUÇÃO
# =======================================================================

visualizar_comparativo_hierarquico(df_comparativo)